In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import joblib
import boto3
import os

In [0]:
#Importando tabela
df = spark.read.table('workspace.gold.recomendation_db_gold')

df_pd = df.toPandas()

#Indices [0...n] para cada id no dataframe
indices = pd.Series(df_pd.index, index=df_pd['id'])

#Indice de cada id do tmdb para ser convertido novamente
indices_tmdb = pd.Series(df_pd['id'], index=df_pd.index)

In [0]:
#Calculando o TF-IDF das 5000 palavras mais importantes.
tfidf = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    lowercase=True,
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df_pd['content'])

In [0]:
#Calcula a similaridade de cosseno entre cada um dos vetores. (O quanto cada filme é similar com todos outros)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [0]:
top_k = 20
n_movies = cosine_sim.shape[0]

top_sim_indices = np.zeros((n_movies, top_k), dtype=np.int32)

#Para cada filme, traz os 20 maiores scores e adiciona ao novo array
for i in range(n_movies):

    sorted_idx = np.argsort(cosine_sim[i])[::-1]
    sorted_idx = sorted_idx[sorted_idx != i]
    top_sim_indices[i] = sorted_idx[:top_k]

cosine_sim = top_sim_indices

In [0]:
#Salva os arquivos
np.save("/Volumes/workspace/artifacts/recommendation_artifacts/cosine_sim.npy", cosine_sim)
indices.to_pickle("/Volumes/workspace/artifacts/recommendation_artifacts/indices.pkl")
indices_tmdb.to_pickle("/Volumes/workspace/artifacts/recommendation_artifacts/ids_tmdb.pkl")


## Testes

In [0]:
%skip


#Encontra o ID do filme pesquisado
def find_id(movie_name):
    movie_name = movie_name.lower()
    movie_id = df_pd.loc[df_pd['title'].str.lower() == movie_name]
    if not movie_id.empty:
        return display(movie_id[['id', 'title']])
    else:
        print('Filme não encontrado.')

In [0]:
%skip


find_id('Saw')


In [0]:
%skip

def recommend(id, n=10):
    idx = indices[id] #Qual indice do id do filme
    movie_indices = cosine_sim[idx][:n] #Top n por indice do id do filme
    return pd.DataFrame(df_pd.iloc[movie_indices]['title'])

In [0]:
%skip


recommend(5255)